# Qwen3-TTS voice cloning on Colab GPU

Generates Nigerian-accented speech in Daniel's voice using Qwen3-TTS. Runs on Colab T4 GPU — much faster than local CPU.

**Runtime → Change runtime type → T4 GPU** before running.

First session: downloads ~5 GB model to your Google Drive. Every future session loads from Drive (no re-download).

## 0. Persistent model cache on Google Drive

Mounts Drive and points HuggingFace cache to a Drive folder. First run downloads the model to Drive. Every subsequent session loads from Drive with no download.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print(f'HuggingFace cache -> {HF_CACHE}')

## 1. Install qwen-tts

In [ ]:
!pip install -q qwen-tts
print('installed')

## 2. Patch min_new_tokens (fixes early-EOS truncation)

The stock qwen-tts hardcodes `min_new_tokens=2`, which causes the model to stop mid-sentence on short inputs. Patch it to accept our value.

In [ ]:
import qwen_tts, os
target = os.path.join(os.path.dirname(qwen_tts.__file__), 'core/models/modeling_qwen3_tts.py')
with open(target) as f: src = f.read()
if '"min_new_tokens": 2,' in src:
    src = src.replace('"min_new_tokens": 2,', '"min_new_tokens": kwargs.pop("min_new_tokens", 2),')
    with open(target, 'w') as f: f.write(src)
    print('patched')
else:
    print('already patched or version mismatch')

## 3. Upload reference WAV and transcript

Uploads once per session. Tip: also drop them into `/content/drive/MyDrive/naija_voice_ref/` so you can copy from Drive instead of uploading each time — but manual upload is fine too.

In [ ]:
from google.colab import files
print('Upload your reference WAV (e.g. my_voice_full.wav)')
uploaded = files.upload()
REF_AUDIO = list(uploaded.keys())[0]
print(f'Ref audio: {REF_AUDIO}')

print('Upload the reference transcript TXT')
uploaded = files.upload()
REF_TEXT_FILE = list(uploaded.keys())[0]
with open(REF_TEXT_FILE) as f: REF_TEXT = f.read().strip()
print(f'Transcript: {REF_TEXT[:120]}...')

## 4. Load model (first run: downloads to Drive ~5 min; subsequent runs: loads from Drive ~30 sec)

In [ ]:
from qwen_tts import Qwen3TTSModel
model = Qwen3TTSModel.from_pretrained('Qwen/Qwen3-TTS-12Hz-1.7B-Base')
print('model loaded')

## 5. Generate (edit TEXT and re-run this cell for different lines)

In [ ]:
import torch, random, numpy as np, soundfile as sf

TEXT = """Hello, my name is Daniel. Let us get into it."""
SEED = 3
TEMPERATURE = 0.7
MIN_TOKENS_PER_CHAR = 15  # forces the model to speak the full sentence
OUT = 'out.wav'

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
min_new = max(80, len(TEXT) * MIN_TOKENS_PER_CHAR)
print(f'Generating ({len(TEXT)} chars, seed={SEED}, min_new_tokens={min_new})')

wavs, sr = model.generate_voice_clone(
    text=TEXT,
    language='English',
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
    temperature=TEMPERATURE,
    min_new_tokens=min_new,
)
wav = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
if hasattr(wav, 'detach'): wav = wav.detach().cpu().numpy()
if wav.ndim == 2 and wav.shape[0] <= 2: wav = wav.T
sf.write(OUT, wav, sr)
print(f'Saved {OUT}: {len(wav)/sr:.1f}s @ {sr}Hz')

from IPython.display import Audio
Audio(OUT)

## 6. Download the result

In [ ]:
from google.colab import files
files.download('out.wav')